In [1]:
# ===== CONFIG =====
import os
INPUT_DIR   = os.environ.get("ITDA_INPUT_DIR",  "./val_images")
OUTPUT_PATH = os.environ.get("ITDA_OUTPUT_PATH", "./submission.csv")
# ==================

# 소비기한 추출 — [CODE]_한정선체고

**구조**: OCR 판독 사다리(`src/ocr/ocr_ladder.py`) → 날짜 판별기(`src/postprocess/date_resolver.py`) → `submission.csv`

- 학습한 가중치 없음. RapidOCR(PP-OCRv4) 사전학습 ONNX + 규칙 기반 후처리
- 모델 16.2MB는 `rapidocr-onnxruntime` 휠에 포함 → **오프라인 채점 환경에서 다운로드 불필요**
- 순수 CPU. 4코어 프로세스 병렬 + 전역 데드라인으로 2400초 상한 방어

설계 근거·실험 기록은 `docs/파이프라인-로직-정리.md` 참고.

In [2]:
# ── 실행 파라미터 (CONFIG 셀과 분리) ───────────────────────────────
# TIME_BUDGET_SEC: 운영진 채점 상한 2400초. 안전 마진 300초를 빼고 운용한다.
TIME_BUDGET_SEC = 2400 - 300
N_WORKERS       = min(4, os.cpu_count() or 1)   # 채점 환경 4-Core

# ONNX Runtime 이 프로세스마다 코어를 전부 잡으려 하면 4개 워커가 서로 뺏어
# 오히려 느려진다. 워커당 1스레드로 묶고 프로세스 단위로만 병렬화한다.
# (이 설정은 재현성에도 필요하다 — 스레드 수가 달라지면 결과가 흔들릴 수 있다)
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "ORT_NUM_THREADS"):
    os.environ[_v] = "1"

import sys, time, csv, glob, traceback
sys.path.insert(0, os.path.abspath("."))        # 레포 루트의 src/ 를 import
T_START = time.time()
print(f"[env] python={sys.version.split()[0]}  cpu={os.cpu_count()}  workers={N_WORKERS}")

[env] python=3.14.3  cpu=10  workers=4


In [3]:
# ── 환경 진단 (실패해도 죽지 않고 로그만 남긴다) ───────────────────
def _diag():
    try:
        import cv2
        print(f"[env] cv2 {cv2.__version__} OK")
    except Exception as e:
        print(f"[env] ⚠️ cv2 import 실패: {e}")
    try:
        import rapidocr_onnxruntime as R
        mp = os.path.join(os.path.dirname(R.__file__), "models")
        n = len(glob.glob(os.path.join(mp, "*.onnx")))
        print(f"[env] rapidocr 번들 모델 {n}개 (오프라인 OK)" if n else
              "[env] ⚠️ rapidocr 번들 모델을 찾지 못함")
    except Exception as e:
        print(f"[env] ⚠️ rapidocr import 실패: {e}")
_diag()

[env] cv2 5.0.0 OK
[env] rapidocr 번들 모델 3개 (오프라인 OK)


In [4]:
# ── 입력 목록 ──────────────────────────────────────────────────────
# 정렬을 고정해야 재현성이 보장된다(운영진 사전 검수 항목).
EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff")
image_files = sorted(
    p for p in glob.glob(os.path.join(INPUT_DIR, "*"))
    if os.path.splitext(p)[1].lower() in EXTS
)
image_ids = [os.path.splitext(os.path.basename(p))[0] for p in image_files]
print(f"[input] {INPUT_DIR} → {len(image_files)}장")

[input] ./val_images → 0장


In [5]:
# ── 워커 로드 ──────────────────────────────────────────────────────
# ⚠️ 워커 함수는 반드시 모듈 파일(src/runner.py)에 있어야 한다.
#    노트북 셀에 정의하면 자식 프로세스가 이름으로 찾지 못해
#    "Can't get attribute '_worker' on <module '__main__'>" 로 죽는다. (실측)
REPO_ROOT = os.path.abspath(".")
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))
sys.path.insert(0, os.path.join(REPO_ROOT, "src", "ocr"))
sys.path.insert(0, os.path.join(REPO_ROOT, "src", "postprocess"))
import runner
runner.init_worker(REPO_ROOT)
print("[env] runner 로드 완료")

[env] runner 로드 완료


In [6]:
# ── 실행 ───────────────────────────────────────────────────────────
def _budget():
    """남은 시간 ÷ 남은 장수. 뒤로 갈수록 빠듯해지면 사다리가 스스로 얕아진다."""
    left = TIME_BUDGET_SEC - (time.time() - T_START)
    return max(1.0, left / max(1, len(image_files)))

rows, n_err, stages = [], 0, {}
if image_files:
    # ⚠️ 장당 예산은 '실제로 몇 개가 동시에 도는가'에 맞춰야 한다.
    #    병렬 예산(×N)을 쓴 채로 순차 폴백되면 예산이 N배 과대평가되어
    #    2400초를 넘긴다. 그래서 폴백 시 예산을 다시 계산한다.
    tasks_par = [(p, i, _budget() * N_WORKERS) for p, i in zip(image_files, image_ids)]
    out = None
    if N_WORKERS > 1 and len(image_files) > 1:
        try:
            import multiprocessing as mp
            # macOS는 fork 시 ObjC 런타임 충돌 → spawn 사용. Linux(채점서버)는 fork.
            # 워커 함수가 src/runner.py 모듈에 있으므로 spawn도 정상 동작한다.
            ctx = mp.get_context("spawn" if sys.platform == "darwin" else "fork")
            with ctx.Pool(N_WORKERS, initializer=runner.init_worker,
                          initargs=(REPO_ROOT,)) as pool:
                out = pool.map(runner.run_one, tasks_par, chunksize=2)
        except Exception as e:
            print(f"[run] ⚠️ 병렬 실패 → 순차로 대체: {e}")
            out = None
    if out is None:                          # 순차 폴백 — 예산을 1배로 다시 계산
        out = []
        for p, i in zip(image_files, image_ids):
            out.append(runner.run_one((p, i, _budget())))

    for iid, y, m, d, fd, stage, sec in out:
        if stage < 0:
            n_err += 1
        stages[stage] = stages.get(stage, 0) + 1
        rows.append({"image_id": iid, "year": y, "month": m, "day": d, "final_date": fd})

elapsed = time.time() - T_START
print(f"[run] {len(rows)}행 | 총 {elapsed:.1f}s | 장당 {elapsed/max(1,len(rows)):.2f}s | 예외 {n_err}건")
print(f"[run] 사다리 단계 분포: {dict(sorted(stages.items()))}")

[run] 0행 | 총 0.1s | 장당 0.10s | 예외 0건
[run] 사다리 단계 분포: {}


In [7]:
# ── 저장 ───────────────────────────────────────────────────────────
FIELDS = ["image_id", "year", "month", "day", "final_date"]

def _norm_cell(v):
    v = "NONE" if v is None else str(v).strip()
    return "NONE" if v.upper() in ("", "NONE", "NAN", "NULL", "-") else v

with open(OUTPUT_PATH, "w", encoding="utf-8", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=FIELDS)   # 인덱스 컬럼 없음
    w.writeheader()
    for r in rows:
        w.writerow({k: _norm_cell(r.get(k)) for k in FIELDS})
print(f"[save] {OUTPUT_PATH}  ({len(rows)}행)")

[save] ./submission.csv  (0행)


In [8]:
# ── 자가 검증 ──────────────────────────────────────────────────────
ok = True
with open(OUTPUT_PATH, encoding="utf-8-sig", newline="") as fh:
    rd = list(csv.reader(fh))
head, body = (rd[0] if rd else []), rd[1:]

def chk(cond, msg):
    global ok
    print(("  ✅ " if cond else "  ❌ ") + msg)
    ok = ok and cond

print("[self-check]")
chk(head == FIELDS,                       f"헤더 == {FIELDS}  (실제 {head})")
chk(len(body) == len(image_files),        f"행 수 {len(body)} == 입력 {len(image_files)}")
chk(all(len(r) == 5 for r in body),       "모든 행이 5개 컬럼")
chk(len(body) > 0,                        "빈 파일 아님")
chk(all(r[4] == "NONE" or (r[4].count("-") == 2)
        for r in body),                   "final_date 형식 YYYY-MM-DD (부분 NONE 포함) 또는 NONE")
chk(elapsed < 2400,                       f"실행 시간 {elapsed:.0f}s < 2400s")
print("[self-check] " + ("ALL PASS" if ok else "FAIL — 제출 전 반드시 수정"))

[self-check]
  ✅ 헤더 == ['image_id', 'year', 'month', 'day', 'final_date']  (실제 ['image_id', 'year', 'month', 'day', 'final_date'])
  ✅ 행 수 0 == 입력 0
  ✅ 모든 행이 5개 컬럼
  ❌ 빈 파일 아님
  ✅ final_date 형식 YYYY-MM-DD (부분 NONE 포함) 또는 NONE
  ✅ 실행 시간 0s < 2400s
[self-check] FAIL — 제출 전 반드시 수정
